In [ ]:
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
import torch

### Loading plots

In [14]:
# Loading file
file_path = "/mnt/lts4/scratch/home/carballo/MechInt/DeFoG/outputs/2025-07-16/14-53-50-sbm-sbm/attention/attn_maps_0_4.pt"

# Load the data with pickle
with open(file_path, "rb") as f:
    data = pickle.load(f)

/mnt/lts4/scratch/home/carballo/MechInt/DeFoG/.pixi/envs/default/lib/python3.11/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torc

In [ ]:
data = data.squeeze(1)   # shape: (L, b = 1, N, N, h) -> (L, N, N, h)
layers, n_nodes, _, heads = data.shape

### Raw maps

In [ ]:
out_dir = "../maps/raw"
os.makedirs(out_dir, exist_ok=True)

In [17]:
for layer in range(layers):
    for head in range(heads):
        attn_map = data[layer, :, :, head].detach().numpy()
        
        plt.figure(figsize=(5, 4))
        sns.heatmap(attn_map, cmap="viridis", cbar=True)
        plt.title(f"Layer {layer+1}, Head {head+1}")
        plt.axis("off")
        
        filename = os.path.join(out_dir, f"layer{layer+1}_head{head+1}.png")
        plt.savefig(filename, bbox_inches="tight", dpi=150)
        plt.close()

### Mean over heads per layer

In [ ]:
out_dir = "../maps/mean"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# Average over heads and plot the heatmap
mean_data = data.mean(dim=1)  # shape: (L, N, N, h) -> (L, N, N)

for layer in range(layers):
    plt.figure(figsize=(10, 8))
    sns.heatmap(mean_data[layer], cmap="viridis", cbar=True)
    plt.title(f"Layer {layer + 1} Attention Map")
    plt.savefig(f"{out_dir}/layer_{layer + 1}_mean.png")
    plt.close()

### Attention rollout / Attention flow

From Quantifying Attention Flow in Transformers (Abnar & Zuidema, 2020)

In [ ]:
out_dir = "../maps/rollout"
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# Average over heads
mean_data = data.mean(dim=1)  # shape: (L, N, N, h) -> (L, N, N)

# Add identity
mean_data = mean_data + torch.eye(mean_data.size(-1), device=mean_data.device)

# Normalize
mean_data = mean_data / mean_data.sum(dim=-1, keepdim=True)

# Rollout: recursive product per layer
# rollout[layer_i] = [product of all previous layers]
rollout = [mean_data[0]]
for layer in range(1, layers):
    rollout.append(rollout[layer - 1] @ mean_data[layer])

# Plot heatmaps
for layer in range(layers):
    plt.figure(figsize=(10, 8))
    sns.heatmap(rollout[layer].detach().cpu().numpy(), cmap="viridis")
    plt.title(f"Layer {layer + 1} Attention Rollout")
    plt.xlabel("Token")
    plt.ylabel("Token")
    plt.show()

In [ ]:
# To get from (L,n,n,h) to (L,N,N), take the min over the heads instead of averaging
min_data = data.min(dim=1).values  # shape: (L, N, N)

# Add identity
min_data = min_data + torch.eye(min_data.size(-1), device=min_data.device)

# Normalize
min_data = min_data / min_data.sum(dim=-1, keepdim=True)

# Repeat process above to get the rollout
rollout = [min_data[0]]
for layer in range(1, layers):
    rollout.append(rollout[layer - 1] @ min_data[layer])


### Markov Chains

### Attention graphs